# DriftWise: Archive of exploration steps
Synthetic stream v1, first stream v2, first gate, and the 40-stream false-update tests. Kept for the record. These are **superseded** by the main notebook (200-stream test, `run_delayed`, 20-seed comparison). Run `DriftWise_Main_clean.ipynb` setup first if you want to rerun any of this.

create the corrected synthetic stream

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_records = 10000
drift_point = 5000

labels_before = (np.random.rand(drift_point) < 0.05).astype(int)
labels_after = (np.random.rand(n_records - drift_point) < 0.20).astype(int)
y_new = np.concatenate([labels_before, labels_after])

X_new = np.zeros((n_records, 2))

for i in range(n_records):
    if i < drift_point:
        attack_mean = 2.0
        normal_mean = 0.0
    else:
        attack_mean = 0.5
        normal_mean = 0.0

    mean = attack_mean if y_new[i] == 1 else normal_mean

    X_new[i] = np.random.normal(loc=[mean, 0], scale=1.0, size=2)

synthetic_df = pd.DataFrame(X_new, columns=["feature_1", "feature_2"])
synthetic_df["Label"] = y_new
synthetic_df["stream_index"] = np.arange(n_records)

print("Stream shape:", synthetic_df.shape)
print("Attack rate before drift:",
      round(synthetic_df.iloc[:drift_point]["Label"].mean(), 3))
print("Attack rate after drift:",
      round(synthetic_df.iloc[drift_point:]["Label"].mean(), 3))

print("\nMean feature_1 by class BEFORE drift:")
print(synthetic_df.iloc[:drift_point].groupby("Label")["feature_1"].mean())

print("\nMean feature_1 by class AFTER drift:")
print(synthetic_df.iloc[drift_point:].groupby("Label")["feature_1"].mean())

train the baseline and watch it break

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import balanced_accuracy_score, recall_score, f1_score

features = ["feature_1", "feature_2"]

# Training data: records 0-3999 (all before the drift)
train = synthetic_df.iloc[0:4000]

# Two test windows, both AFTER the training data
test_before = synthetic_df.iloc[4000:5000]   # still before the drift
test_after = synthetic_df.iloc[5000:6000]    # after the drift

# Fit the scaler on training data only, to avoid leaking information
scaler = StandardScaler()
X_train = scaler.fit_transform(train[features])
y_train = train["Label"]

model = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

def evaluate(name, window):
    X = scaler.transform(window[features])
    y = window["Label"]
    pred = model.predict(X)
    print(f"--- {name} ---")
    print("Attacks in window:", int(y.sum()), "of", len(y))
    print("Balanced accuracy:", round(balanced_accuracy_score(y, pred), 3))
    print("Normal recall:", round(recall_score(y, pred, pos_label=0), 3))
    print("Attack recall:", round(recall_score(y, pred, pos_label=1), 3))
    print("Macro-F1:", round(f1_score(y, pred, average="macro"), 3))
    print()

evaluate("BEFORE drift (records 4000-4999)", test_before)
evaluate("AFTER drift (records 5000-5999)", test_after)

watch the errors over time

In [ ]:
import matplotlib.pyplot as plt

# Replay the stream from record 4000 onward using the frozen model
stream = synthetic_df.iloc[4000:].copy()
stream["pred"] = model.predict(scaler.transform(stream[features]))
stream["error"] = (stream["pred"] != stream["Label"]).astype(int)

# Rolling error rate over the last 200 records
stream["rolling_error"] = stream["error"].rolling(200).mean()

# Rolling attack recall over the last 40 attacks
attacks = stream[stream["Label"] == 1].copy()
attacks["caught"] = (attacks["pred"] == 1).astype(int)
attacks["rolling_attack_recall"] = attacks["caught"].rolling(40).mean()

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(stream["stream_index"], stream["rolling_error"])
axes[0].axvline(5000, color="red", linestyle="--", label="drift point")
axes[0].set_ylabel("Error rate (last 200)")
axes[0].legend()

axes[1].plot(attacks["stream_index"], attacks["rolling_attack_recall"])
axes[1].axvline(5000, color="red", linestyle="--")
axes[1].set_ylabel("Attack recall (last 40 attacks)")
axes[1].set_xlabel("Stream index")

plt.tight_layout()
plt.show()

print("Error rate, records 4000-4999:", round(stream.iloc[:1000]["error"].mean(), 3))
print("Error rate, records 5000-9999:", round(stream.iloc[1000:]["error"].mean(), 3))

In [ ]:
import os
import pandas as pd

base = "/content/drive/MyDrive/DriftWise"
stream_path = os.path.join(base, "data/processed/synthetic_stream_v1.csv")
plot_path = os.path.join(base, "results/plots/baseline_frozen_model_errors.png")

# Save the synthetic stream and the plot from the last step
synthetic_df.to_csv(stream_path, index=False)
fig.savefig(plot_path, dpi=150, bbox_inches="tight")

# Check that the saved files can be read back
check = pd.read_csv(stream_path)
print("Stream saved:", stream_path)
print("Rows and columns read back:", check.shape)
print("Plot saved:", os.path.exists(plot_path))

add ADWIN and see if it detects the drift

In [ ]:
from river import drift
import matplotlib.pyplot as plt

# Uses the "stream" table from Step 3 (records 4000 onward, with an "error" column)
adwin = drift.ADWIN(delta=0.002)   # default sensitivity
alarms = []

for idx, err in zip(stream["stream_index"], stream["error"]):
    out = adwin.update(int(err))
    # Different river versions report drift in different ways
    detected = out[0] if isinstance(out, tuple) else adwin.drift_detected
    if detected:
        alarms.append(int(idx))

true_drift = 5000
false_alarms = [a for a in alarms if a < true_drift]
after = [a for a in alarms if a >= true_drift]

print("All alarms at stream index:", alarms)
print("Alarms BEFORE the true drift (false alarms):", false_alarms)
if after:
    print("First alarm after the drift:", after[0])
    print("Detection delay (records):", after[0] - true_drift)
else:
    print("ADWIN did not detect the drift.")

plt.figure(figsize=(10, 3.5))
plt.plot(stream["stream_index"], stream["rolling_error"])
plt.axvline(true_drift, color="red", linestyle="--", label="true drift")
for a in alarms:
    plt.axvline(a, color="green", alpha=0.6)
plt.plot([], [], color="green", label="ADWIN alarm")
plt.xlabel("Stream index")
plt.ylabel("Error rate (last 200)")
plt.legend()
plt.show()

check whether there is enough evidence to adapt

In [ ]:
from river import drift

# Re-run ADWIN and capture the moment it fires
adwin = drift.ADWIN(delta=0.002)
alarm_index, width_at_alarm = None, None

for idx, err in zip(stream["stream_index"], stream["error"]):
    out = adwin.update(int(err))
    detected = out[0] if isinstance(out, tuple) else adwin.drift_detected
    if detected:
        alarm_index = int(idx)
        try:
            width_at_alarm = int(adwin.width)   # size of the recent window ADWIN kept
        except Exception:
            width_at_alarm = 600                # fallback if this river version lacks .width
        break

change_estimate = alarm_index - width_at_alarm
print("Alarm at:", alarm_index)
print("Estimated change point:", change_estimate, "(true drift is 5000, which we know only because this is synthetic)")

# Labeled records after the estimated change, up to the alarm
recent = stream[(stream["stream_index"] > change_estimate) &
                (stream["stream_index"] <= alarm_index)]

n = len(recent)
train_end = int(n * 0.6)
gap = 50

train_part = recent.iloc[:train_end]
val_part = recent.iloc[train_end + gap:]

print("\nRecent labeled records available:", n)
print("Train:      ", len(train_part), "records,", int(train_part["Label"].sum()), "attacks")
print("Gap skipped:", gap, "records")
print("Validation: ", len(val_part), "records,", int(val_part["Label"].sum()), "attacks")

print("\nEvidence check (provisional thresholds from our spec):")
print("Validation has at least 200 records:", len(val_part) >= 200)
print("Validation has at least 30 attacks: ", int(val_part["Label"].sum()) >= 30)

In [ ]:
alarm_index = 5599
true_drift = 5000          # known only because this is a synthetic stream

labeled = stream[stream["stream_index"] <= alarm_index]   # labels available at the alarm

val_size = 250
gap = 50
short_train = 300
long_train = 600

val = labeled.iloc[-val_size:]
train_end = len(labeled) - val_size - gap
train_short = labeled.iloc[train_end - short_train : train_end]
train_long = labeled.iloc[train_end - long_train : train_end]

def describe(name, df):
    post = (df["stream_index"] >= true_drift).mean() * 100
    print(f"{name}: records {df['stream_index'].min()}-{df['stream_index'].max()}, "
          f"{len(df)} records, {int(df['Label'].sum())} attacks, "
          f"{post:.0f}% truly post-drift")

describe("Short train window", train_short)
describe("Long train window ", train_long)
print(f"(gap of {gap} records skipped)")
describe("Validation        ", val)

print("\nValidation has at least 200 records:", len(val) >= 200)
print("Validation has at least 30 attacks: ", int(val["Label"].sum()) >= 30)

train two candidate models and compare them with the current one

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import balanced_accuracy_score, recall_score, f1_score

features = ["feature_1", "feature_2"]
X_val_raw = val[features]
y_val = val["Label"]

def train_candidate(train_df):
    sc = StandardScaler()
    X = sc.fit_transform(train_df[features])
    m = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
    m.fit(X, train_df["Label"])
    return sc, m

def score(sc, m):
    pred = m.predict(sc.transform(X_val_raw))
    normal_recall = recall_score(y_val, pred, pos_label=0)
    attack_recall = recall_score(y_val, pred, pos_label=1)
    return {
        "macro_F1": f1_score(y_val, pred, average="macro"),
        "balanced_acc": balanced_accuracy_score(y_val, pred),
        "normal_recall": normal_recall,
        "attack_recall": attack_recall,
        "worst_class_recall": min(normal_recall, attack_recall),
    }

sc_short, m_short = train_candidate(train_short)
sc_long, m_long = train_candidate(train_long)

results = pd.DataFrame({
    "current model": score(scaler, model),   # original model and scaler from Step 2
    "candidate: short window": score(sc_short, m_short),
    "candidate: long window": score(sc_long, m_long),
}).T.round(3)

display(results)

print("\nDifference from the current model (candidate minus current):")
display((results.loc[["candidate: short window", "candidate: long window"]]
         - results.loc["current model"]).round(3))

build a stream where adaptation can help

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import balanced_accuracy_score, recall_score, f1_score

rng = np.random.default_rng(42)
n_records, drift_point = 10000, 5000

# Attack rate: about 5% before the drift, about 20% after
p_attack = np.where(np.arange(n_records) < drift_point, 0.05, 0.20)
y2 = (rng.random(n_records) < p_attack).astype(int)

# Start with pure noise, then add the attack signal
X2 = rng.normal(size=(n_records, 2))
X2[:drift_point, 0] += 2.0 * y2[:drift_point]      # BEFORE drift: signal in feature_1
X2[drift_point:, 1] += 2.0 * y2[drift_point:]      # AFTER drift: signal moves to feature_2

synthetic_v2 = pd.DataFrame(X2, columns=["feature_1", "feature_2"])
synthetic_v2["Label"] = y2
synthetic_v2["stream_index"] = np.arange(n_records)

synthetic_v2.to_csv(
    "/content/drive/MyDrive/DriftWise/data/processed/synthetic_stream_v2.csv",
    index=False)

feats = ["feature_1", "feature_2"]

def fit_model(df):
    sc = StandardScaler()
    m = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
    m.fit(sc.fit_transform(df[feats]), df["Label"])
    return sc, m

def report(name, sc, m, df):
    pred = m.predict(sc.transform(df[feats]))
    y = df["Label"]
    print(f"{name}: attacks={int(y.sum())}, "
          f"balanced_acc={balanced_accuracy_score(y, pred):.3f}, "
          f"attack_recall={recall_score(y, pred, pos_label=1):.3f}, "
          f"normal_recall={recall_score(y, pred, pos_label=0):.3f}, "
          f"macro_F1={f1_score(y, pred, average='macro'):.3f}")

old_sc, old_m = fit_model(synthetic_v2.iloc[0:4000])
print("Old model (trained on records 0-3999):")
report("  before drift (4000-4999)", old_sc, old_m, synthetic_v2.iloc[4000:5000])
report("  after drift  (5500-5999)", old_sc, old_m, synthetic_v2.iloc[5500:6000])

new_sc, new_m = fit_model(synthetic_v2.iloc[5000:5500])
print("\nModel retrained on records 5000-5499 (after drift):")
report("  after drift  (5500-5999)", new_sc, new_m, synthetic_v2.iloc[5500:6000])

run ADWIN on the new stream

In [ ]:
from river import drift
import matplotlib.pyplot as plt

# Replay stream v2 from record 4000 onward with the frozen old model
stream2 = synthetic_v2.iloc[4000:].copy()
stream2["pred"] = old_m.predict(old_sc.transform(stream2[feats]))
stream2["error"] = (stream2["pred"] != stream2["Label"]).astype(int)
stream2["rolling_error"] = stream2["error"].rolling(200).mean()

adwin = drift.ADWIN(delta=0.002)
alarms = []

for idx, err in zip(stream2["stream_index"], stream2["error"]):
    out = adwin.update(int(err))
    detected = out[0] if isinstance(out, tuple) else adwin.drift_detected
    if detected:
        alarms.append(int(idx))

true_drift = 5000
false_alarms = [a for a in alarms if a < true_drift]
after = [a for a in alarms if a >= true_drift]

print("All alarms at stream index:", alarms)
print("Alarms BEFORE the true drift (false alarms):", false_alarms)
if after:
    print("First alarm after the drift:", after[0])
    print("Detection delay (records):", after[0] - true_drift)
else:
    print("ADWIN did not detect the drift.")

print("\nError rate, records 4000-4999:", round(stream2.iloc[:1000]["error"].mean(), 3))
print("Error rate, records 5000-9999:", round(stream2.iloc[1000:]["error"].mean(), 3))

plt.figure(figsize=(10, 3.5))
plt.plot(stream2["stream_index"], stream2["rolling_error"])
plt.axvline(true_drift, color="red", linestyle="--", label="true drift")
for a in alarms:
    plt.axvline(a, color="green", alpha=0.6)
plt.plot([], [], color="green", label="ADWIN alarm")
plt.xlabel("Stream index")
plt.ylabel("Error rate (last 200)")
plt.legend()
plt.show()

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import balanced_accuracy_score, recall_score

true_drift = 5000
alarm_index = 5375
val_size, gap, train_size = 250, 50, 300

def worst_class_recall(y, pred):
    return min(recall_score(y, pred, pos_label=0),
               recall_score(y, pred, pos_label=1))

rows = []
for t in range(alarm_index, alarm_index + 800, 100):
    labeled = synthetic_v2.iloc[: t + 1]          # only labels available up to time t

    val = labeled.iloc[-val_size:]
    train_end = len(labeled) - val_size - gap
    train = labeled.iloc[train_end - train_size : train_end]

    # Train a candidate on the recent training window
    sc = StandardScaler()
    cand = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
    cand.fit(sc.fit_transform(train[feats]), train["Label"])

    y = val["Label"]
    pred_old = old_m.predict(old_sc.transform(val[feats]))
    pred_new = cand.predict(sc.transform(val[feats]))

    rows.append({
        "checkpoint": t,
        "train % post-drift": round((train["stream_index"] >= true_drift).mean() * 100),
        "val attacks": int(y.sum()),
        "old: bal_acc": round(balanced_accuracy_score(y, pred_old), 3),
        "new: bal_acc": round(balanced_accuracy_score(y, pred_new), 3),
        "old: worst recall": round(worst_class_recall(y, pred_old), 3),
        "new: worst recall": round(worst_class_recall(y, pred_new), 3),
    })

display(pd.DataFrame(rows))

build the acceptance gate

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

rng = np.random.default_rng(0)

def fast_metrics(y, p):
    f1s, recalls = [], []
    for c in (0, 1):
        tp = np.sum((y == c) & (p == c))
        fp = np.sum((y != c) & (p == c))
        fn = np.sum((y == c) & (p != c))
        denom = 2 * tp + fp + fn
        f1s.append(2 * tp / denom if denom > 0 else 0.0)
        recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
    return {
        "macro_f1": np.mean(f1s),
        "bal_acc": np.mean(recalls),
        "worst_recall": min(recalls),
    }

def lower_bounds(y, p_new, p_old, n_boot=1000):
    n = len(y)
    diffs = {"macro_f1": [], "bal_acc": [], "worst_recall": []}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        m_new = fast_metrics(y[idx], p_new[idx])
        m_old = fast_metrics(y[idx], p_old[idx])
        for k in diffs:
            diffs[k].append(m_new[k] - m_old[k])
    return {k: np.percentile(v, 2.5) for k, v in diffs.items()}

rows = []
for t in range(alarm_index, alarm_index + 800, 100):
    labeled = synthetic_v2.iloc[: t + 1]
    val = labeled.iloc[-val_size:]
    train_end = len(labeled) - val_size - gap
    train = labeled.iloc[train_end - train_size : train_end]

    sc = StandardScaler()
    cand = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
    cand.fit(sc.fit_transform(train[feats]), train["Label"])

    y = val["Label"].to_numpy()
    p_old = old_m.predict(old_sc.transform(val[feats]))
    p_new = cand.predict(sc.transform(val[feats]))
    lb = lower_bounds(y, p_new, p_old)

    cond_A = (len(val) >= 200) and (int(y.sum()) >= 30)
    cond_B = lb["macro_f1"] > -0.01
    cond_C = lb["worst_recall"] > -0.03
    cond_D = lb["bal_acc"] > 0

    if not cond_A:
        decision = "DEFER"
    elif cond_B and cond_C and cond_D:
        decision = "ACCEPT"
    else:
        decision = "REJECT"

    failed = [name for name, ok in zip("BCD", [cond_B, cond_C, cond_D]) if not ok]
    rows.append({
        "checkpoint": t,
        "train % post-drift": round((train["stream_index"] >= 5000).mean() * 100),
        "lower bound dMacroF1": round(lb["macro_f1"], 3),
        "lower bound dWorstRecall": round(lb["worst_recall"], 3),
        "lower bound dBalAcc": round(lb["bal_acc"], 3),
        "decision": decision,
        "failed": ",".join(failed) if decision == "REJECT" else "",
    })

gate = pd.DataFrame(rows)
display(gate)

accepted = gate[gate["decision"] == "ACCEPT"]
if len(accepted):
    first = int(accepted.iloc[0]["checkpoint"])
    print("First ACCEPT at record:", first, "| delay after true drift:", first - 5000)
else:
    print("No candidate was accepted.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

def make_no_drift_stream(seed, n=6000):
    r = np.random.default_rng(seed)
    y = (r.random(n) < 0.05).astype(int)
    X = r.normal(size=(n, 2))
    X[:, 0] += 2.0 * y                      # same signal the whole time: no drift
    return pd.DataFrame({"feature_1": X[:, 0], "feature_2": X[:, 1], "Label": y})

val_size, gap, train_size = 1000, 50, 300
decisions = []

for seed in range(40):
    s = make_no_drift_stream(seed)

    # Current model: trained on the first 4000 records
    sc_old = StandardScaler()
    m_old = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
    m_old.fit(sc_old.fit_transform(s.iloc[:4000][feats]), s.iloc[:4000]["Label"])

    # Candidate: trained on 300 recent records, then validated on the newest 1000
    val = s.iloc[-val_size:]
    train_end = len(s) - val_size - gap
    train = s.iloc[train_end - train_size : train_end]
    sc_new = StandardScaler()
    m_new = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
    m_new.fit(sc_new.fit_transform(train[feats]), train["Label"])

    y = val["Label"].to_numpy()
    p_old = m_old.predict(sc_old.transform(val[feats]))
    p_new = m_new.predict(sc_new.transform(val[feats]))
    lb = lower_bounds(y, p_new, p_old)

    cond_A = int(y.sum()) >= 30
    cond_B = lb["macro_f1"] > -0.01
    cond_C = lb["worst_recall"] > -0.03
    cond_D = lb["bal_acc"] > 0

    if not cond_A:
        decisions.append("DEFER")
    elif cond_B and cond_C and cond_D:
        decisions.append("ACCEPT")
    else:
        decisions.append("REJECT")

print(pd.Series(decisions).value_counts())
print("\nFalse update rate (ACCEPT out of 40 no-drift streams):",
      decisions.count("ACCEPT") / 40)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

val_size, gap, train_size = 1000, 50, 1000
checkpoints = list(range(4500, 6000, 200))      # 8 checkpoints per stream

streams_with_accept = 0
total_accepts = 0
total_checks = 0

for seed in range(40):
    s = make_no_drift_stream(seed)               # 6000 records, no drift

    # Current model: trained on the first 3000 records
    sc_old = StandardScaler()
    m_old = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
    m_old.fit(sc_old.fit_transform(s.iloc[:3000][feats]), s.iloc[:3000]["Label"])

    accepted_here = False
    for t in checkpoints:
        labeled = s.iloc[: t + 1]
        val = labeled.iloc[-val_size:]
        train_end = len(labeled) - val_size - gap
        train = labeled.iloc[train_end - train_size : train_end]

        sc_new = StandardScaler()
        m_new = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)
        m_new.fit(sc_new.fit_transform(train[feats]), train["Label"])

        y = val["Label"].to_numpy()
        p_old = m_old.predict(sc_old.transform(val[feats]))
        p_new = m_new.predict(sc_new.transform(val[feats]))
        lb = lower_bounds(y, p_new, p_old)

        total_checks += 1
        enough = int(y.sum()) >= 30
        if enough and lb["macro_f1"] > -0.01 and lb["worst_recall"] > -0.03 and lb["bal_acc"] > 0:
            total_accepts += 1
            accepted_here = True

    streams_with_accept += int(accepted_here)

print("Gate checks run:", total_checks)
print("ACCEPT decisions (all false updates):", total_accepts)
print("False update rate per check:", round(total_accepts / total_checks, 3))
print("Streams with at least one false update:", streams_with_accept, "of 40")
print("False update rate per stream:", round(streams_with_accept / 40, 3))